### SMS Spam Detection

<p>You are hired as an AI expert in the development department of a telecommunications company. The first thing on your orientation plan is a small project that your boss has assigned you for the following given situation. Your supervisor has given away his private cell phone number on too many websites and is now complaining about daily spam SMS. Therefore, it is your job to write a spam detector in Python. </p>

<p>In doing so, you need to use a Naive Bayes classifier that can handle both bag-of-words (BoW) and tf-idf features as input. For the evaluation of your spam detector, an SMS collection is available as a dataset - this has yet to be suitably split into train and test data. To keep the costs as low as possible and to avoid problems with copyrights, your boss insists on a new development with Python.</p>

<p>Include a short description of the data preprocessing steps, method, experiment design, hyper-parameters, and evaluation metric. Also, document your findings, drawbacks, and potential improvements.</p>

<p>Note: You need to implement the bag-of-words (BoW) and tf-idf feature extractor from scratch. You can use existing python libraries for other tasks.</p>

**Dataset and Resources**

* SMS Spam Collection Dataset: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

In [ ]:

import numpy as np 
from collections import Counter  
from sklearn.model_selection import train_test_split  
from sklearn.naive_bayes import MultinomialNB  
from sklearn.metrics import classification_report  
import nltk  
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string  


nltk.download('stopwords', quiet=True)
# Stopwörter sind sehr häufige Wörter (wie "the", "and"), die entfernt werden
STOP_WORDS = set(stopwords.words("english"))
# Stemmer reduziert Wörter auf ihren Wortstamm (z.B. "running" -> "run")
STEMMER = PorterStemmer()

#(Preprocessing)
def preprocess_text(documents):
   
    cleaned_documents = []
    for text in documents:
        # Kleinschreibung
        text = text.lower()
        #  Satzzeichen durch Leerzeichen ersetzen
        for p in string.punctuation:
            text = text.replace(p, " ")
        # Ziffern entfernen, erzeugt einen Text ohne Zahlen
        result_chars = []
        for char in text:
            if not char.isdigit():
                result_chars.append(char)
        text = ''.join(result_chars)     
        #  Tokenisieren 
        words = text.split()
        # Stopwords entfernen und Stemming anwenden
        stemmed_words = []
        for w in words:
            if w not in STOP_WORDS:
                stemmed_words.append(STEMMER.stem(w))
        words = stemmed_words
        #  bereinigten Text als String zurückgeben 
        cleaned_documents.append(' '.join(words))
    return cleaned_documents


def bag_of_words(documents, vocab=None):
    
    # aus den Dokumenten ein Vokabular erstellen
    if vocab is None:
        vocab, index = {}, 0
        for doc in documents:
            for w in doc.split():
                if w not in vocab:
                    vocab[w] = index
                    index += 1
    # Numpy-Matrix mit Nullen: Zeilen = Dokumente, Spalten = Vokabulargröße
    bow = np.zeros((len(documents), len(vocab)), dtype=float)
    # Matrix mit Wortzählungen pro Dokument füllen
    for i, doc in enumerate(documents):
        counts = Counter(doc.split())  # zählt, wie oft jedes Wort im Dokument vorkommt
        for w, c in counts.items():
            j = vocab.get(w)
            if j is not None:
                bow[i, j] = c
    return bow, vocab


def tf_idf(documents, vocab=None):

    bow, vocab = bag_of_words(documents, vocab) #Bag-of-Words-Matrix berechne
    N = bow.shape[0]  # Anzahl Dokumente
    df = (bow > 0).sum(axis=0) # Dokumentenfrequenz
    idf = np.log((1.0 + N) / (1.0 + df)) + 1.0  # Formel: log(Gesamtdokumente / Dokumentenfrequenz), smoothed
    row_sums = bow.sum(axis=1, keepdims=True) # Term Frequency (TF)                     //  vgl. ChatGPT
    tf = np.divide(bow, row_sums, where=row_sums != 0)  # vermeide Division durch 0     //  vgl. ChatGPT
    tfidf = tf * idf    # TF-IDF: TF multipliziert mit IDF 
    return tfidf, vocab

def spam_detector(filepath):
   
    messages = []
    labels = []
    
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            label, text = line.strip().split('\t', 1) # Teilt String an Tabulator, nur einmal splitten, erste Split‑Ergebnis zu label, zweite zu text
            labels.append(1 if label == "spam" else 0)
            messages.append(text)
    
    # Preprocessing
    cleaned_messages = preprocess_text(messages)
    
    # Train-Test-Split     
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        cleaned_messages, labels, test_size=0.2, random_state=42   # vgl. ChatGPT
import numpy as np 
from collections import Counter  
from sklearn.model_selection import train_test_split  
from sklearn.naive_bayes import MultinomialNB  
from sklearn.metrics import classification_report  
import nltk  
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string  


nltk.download('stopwords', quiet=True)
# Stopwörter sind sehr häufige Wörter (wie "the", "and"), die entfernt werden
STOP_WORDS = set(stopwords.words("english"))
# Stemmer reduziert Wörter auf ihren Wortstamm (z.B. "running" -> "run")
STEMMER = PorterStemmer()

#(Preprocessing)
def preprocess_text(documents):
   
    cleaned_documents = []
    for text in documents:
        # Kleinschreibung
        text = text.lower()
        #  Satzzeichen durch Leerzeichen ersetzen
        for p in string.punctuation:
            text = text.replace(p, " ")
        # Ziffern entfernen, erzeugt einen Text ohne Zahlen
        result_chars = []
        for char in text:
            if not char.isdigit():
                result_chars.append(char)
        text = ''.join(result_chars)     
        #  Tokenisieren 
        words = text.split()
        # Stopwords entfernen und Stemming anwenden
        stemmed_words = []
        for w in words:
            if w not in STOP_WORDS:
                stemmed_words.append(STEMMER.stem(w))
        words = stemmed_words
        #  bereinigten Text als String zurückgeben 
        cleaned_documents.append(' '.join(words))
    return cleaned_documents


def bag_of_words(documents, vocab=None):
    
    # aus den Dokumenten ein Vokabular erstellen
    if vocab is None:
        vocab, index = {}, 0
        for doc in documents:
            for w in doc.split():
                if w not in vocab:
                    vocab[w] = index
                    index += 1
    # Numpy-Matrix mit Nullen: Zeilen = Dokumente, Spalten = Vokabulargröße
    bow = np.zeros((len(documents), len(vocab)), dtype=float)
    # Matrix mit Wortzählungen pro Dokument füllen
    for i, doc in enumerate(documents):
        counts = Counter(doc.split())  # zählt, wie oft jedes Wort im Dokument vorkommt
        for w, c in counts.items():
            j = vocab.get(w)
            if j is not None:
                bow[i, j] = c
    return bow, vocab


def tf_idf(documents, vocab=None):

    bow, vocab = bag_of_words(documents, vocab) #Bag-of-Words-Matrix berechne
    N = bow.shape[0]  # Anzahl Dokumente
    df = (bow > 0).sum(axis=0) # Dokumentenfrequenz
    idf = np.log((1.0 + N) / (1.0 + df)) + 1.0  # Formel: log(Gesamtdokumente / Dokumentenfrequenz), smoothed
    row_sums = bow.sum(axis=1, keepdims=True) # Term Frequency (TF)                     //  vgl. ChatGPT
    tf = np.divide(bow, row_sums, where=row_sums != 0)  # vermeide Division durch 0     //  vgl. ChatGPT
    tfidf = tf * idf    # TF-IDF: TF multipliziert mit IDF 
    return tfidf, vocab

def spam_detector(filepath):
   
    messages = []
    labels = []
    
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            label, text = line.strip().split('\t', 1) # Teilt String an Tabulator, nur einmal splitten, erste Split‑Ergebnis zu label, zweite zu text
            labels.append(1 if label == "spam" else 0)
            messages.append(text)
    
    # Preprocessing
    cleaned_messages = preprocess_text(messages)
    
    # Train-Test-Split     
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        cleaned_messages, labels, test_size=0.2, random_state=42   # vgl. ChatGPT
    )
    
    # Bag-of-Words erstellen
    print("Bag-of-Words-Modellierung:")
    bow_train, vocab_bow = bag_of_words(train_texts) # Vokabular aus Trainings‑SMS und BoW Matrix erstellen
    bow_test, _ = bag_of_words(test_texts, vocab_bow) # gleiche Vokabular wie von Trainingsdaten
    
    # Naive Bayes auf Bag-of-Words trainieren   vgl. https://www.geeksforgeeks.org/deep-learning/sms-spam-detection-using-tensorflow-in-python/
    model_bow = MultinomialNB()
    model_bow.fit(bow_train, train_labels) # Trainiert das Naive‑Bayes‑Modell an die Trainingsdaten
    predictions_bow = model_bow.predict(bow_test) # Verwendet trainierte Modell, um Vorhersagen auf Testdaten zu machen
    print(classification_report(test_labels, predictions_bow, target_names=["Ham", "Spam"])) 
   
    # TF-IDF erstellen
    print("\nTF-IDF-Modellierung:")
    tfidf_train, vocab_tfidf = tf_idf(train_texts)
    tfidf_test, _ = tf_idf(test_texts, vocab_tfidf)
    
    # Naive Bayes auf TF-IDF trainieren  vgl. https://www.geeksforgeeks.org/deep-learning/sms-spam-detection-using-tensorflow-in-python/
    model_tfidf = MultinomialNB()
    model_tfidf.fit(tfidf_train, train_labels)
    predictions_tfidf = model_tfidf.predict(tfidf_test)
    print(classification_report(test_labels, predictions_tfidf, target_names=["Ham", "Spam"]))

spam_detector(r"C:\Users\herma\Desktop\NLP\SMSSpamCollection")

    )
    
    # Bag-of-Words erstellen
    print("Bag-of-Words-Modellierung:")
    bow_train, vocab_bow = bag_of_words(train_texts) # Vokabular aus Trainings‑SMS und BoW Matrix erstellen
    bow_test, _ = bag_of_words(test_texts, vocab_bow) # gleiche Vokabular wie von Trainingsdaten
    
    # Naive Bayes auf Bag-of-Words trainieren   vgl. https://www.geeksforgeeks.org/deep-learning/sms-spam-detection-using-tensorflow-in-python/
    model_bow = MultinomialNB()
    model_bow.fit(bow_train, train_labels) # Trainiert das Naive‑Bayes‑Modell an die Trainingsdaten
    predictions_bow = model_bow.predict(bow_test) # Verwendet trainierte Modell, um Vorhersagen auf Testdaten zu machen
    print(classification_report(test_labels, predictions_bow, target_names=["Ham", "Spam"])) 
   
    # TF-IDF erstellen
    print("\nTF-IDF-Modellierung:")
    tfidf_train, vocab_tfidf = tf_idf(train_texts)
    tfidf_test, _ = tf_idf(test_texts, vocab_tfidf)
    
    # Naive Bayes auf TF-IDF trainieren  vgl. https://www.geeksforgeeks.org/deep-learning/sms-spam-detection-using-tensorflow-in-python/
    model_tfidf = MultinomialNB()
    model_tfidf.fit(tfidf_train, train_labels)
    predictions_tfidf = model_tfidf.predict(tfidf_test)
    print(classification_report(test_labels, predictions_tfidf, target_names=["Ham", "Spam"]))

spam_detector(r"C:\Users\herma\Desktop\NLP\SMSSpamCollection")


Bag-of-Words-Modellierung:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       954
        Spam       0.94      0.96      0.95       161

    accuracy                           0.98      1115
   macro avg       0.97      0.97      0.97      1115
weighted avg       0.98      0.98      0.98      1115


TF-IDF-Modellierung:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       954
        Spam       0.90      0.93      0.91       161

    accuracy                           0.97      1115
   macro avg       0.94      0.96      0.95      1115
weighted avg       0.98      0.97      0.98      1115



## Dokumentation der Lösung

### 1. Datenvorverarbeitung (Preprocessing )
- Kleinschreibung: alle Texte in Lowercase umwandeln
- Satzzeichen-Entfernung: Alle Satzzeichen durch Leerzeichen ersetzen
- Ziffern-Entfernung: Alle numerischen Zeichen entfernen
- Tokenisierung: Texte in Wörter aufteilen (Split bei Whitespace)
- Stopwords-Entfernung: Häufige englische Wörter (z.B. "the", "and") entfernen
- Stemming: PorterStemmer reduziert Wörter auf ihre Wurzel (z.B. "running" → "run")

### 2. Methode
- **Klassifikationsmodell:** Multinomial Naive Bayes 
- **Darstellung der Texte:** 
  - Bag-of-Words (BoW): Zählt Wort-Häufigkeiten pro Dokument
  - TF-IDF: Gewichtet Wörter basierend auf ihrer Seltenheit

### 3. Experimentdesign
- **Train/Test Split:** 80% Training, 20% Testing
- **Vokabular:** Erstellt aus Trainingsdaten, wird auf Testdaten angewendet (gleiche Spalten)

### 4. Hyperparameter
- `test_size=0.2`: 20% der Daten für Testing
- `random_state=42`: Seed für Zufälligkeit
- `alpha=1.0` (MultinomialNB): Laplace-Smoothing verhindert Zero-Wahrscheinlichkeiten

### 5. Evaluierungsmetrik
- **Precision:** Wie viele der als "Spam" klassifizierten SMS sind wirklich Spam?
- **Recall:** Wie viele echter Spam-SMS werden erkannt?
- **F1-Score:** Harmonisches Mittel aus Precision und Recall
- **Support:** Anzahl der Instanzen pro Klasse in den Testdaten

### 6. Erkenntnisse
Das Bag-of-Words Modell erzielt eine Genauigkeit von 98% und übertrifft damit das TF-IDF Modell (97%). Bei der Spam-Erkennung erreicht BoW eine Precision von 0.94 und einen Recall von 0.96, während TF-IDF eine Precision von 0.90 und einen Recall von 0.93 erreicht. Das bedeutet: Bag-of-Words erkennt echten Spam zuverlässiger. Bei Ham-SMS zeigen beide Modelle exzellente Ergebnisse (Precision 0.99 für BoW, 0.99 für TF-IDF). Der Datensatz ist mit 954 Ham-SMS und 161 Spam-SMS unausgewogen (85% zu 15%), beide Modelle funktionieren aber trotzdem gut.

### 7. Nachteile 
- **PorterStemmer Aggressivität:** Der englische PorterStemmer könnte für SMS-Sprache zu aggressiv sein und wichtige Unterscheidungsmerkmale verlieren
- **Stopword-Entfernung:** Das Entfernen von Stopwörtern könnte SMS-spezifische Spam-Signale entfernen, die in diesen Wörtern versteckt sind
- **Einfache Tokenisierung:** Whitespace-basiertes Splitting ignoriert Emoji, Sonderzeichen und URL-Patterns, die oft in Spam vorkommen
- **Begrenzte Features:** BoW/TF-IDF ignorieren Wortbedeutung, Kontext und Wortabfolgen, nur die Häufigkeiten zählen
- **Vokabular-Limitierung:** Neue Wörter in den Testdaten können nicht verarbeitet werden, da das Vokabular nur aus Trainingsdaten erstellt wird

### 8. Mögliche Verbesserungen
- Verschiedene Stemmer testen oder kein Stemming verwenden, um mehr Wortinformation zu bewahren
- Stopword-Liste anpassen: SMS-spezifische Wörter identifizieren und beibehalten, die für Spam-Erkennung wichtig sind
- N-Grams verwenden (Bigrams, Trigrams) um Wortsequenzen und Kontext besser zu erfassen
- Großbuchstaben-Anteil, Sonderzeichen-Häufigkeit, Wort-Länge als zusätzliche Features verwenden
